### Gold Layer — Stock Price Analytics (Materialized Views)

This notebook defines the Gold layer of the Medallion Architecture for the stock prices pipeline using Databricks SQL Materialized Views.

It reads from the clean stocks_silver table and produces three analytical views ready for dashboards and business consumption.

**stock_daily_returns**

Calculates the daily percentage change in closing price for each ticker. Uses the LAG window function to look back one trading day per ticker, computes the return as (today - yesterday) / yesterday × 100, and rounds to 4 decimal places. This is the foundational metric for performance tracking.

**stocks_moving_averages**

Computes 7-day and 30-day rolling averages of the closing price for each ticker using sliding window functions. The 7-day average captures short-term price momentum while the 30-day average smooths out noise to reveal the longer-term trend. Used for trend analysis and trading signal generation.

**stocks_volatility**

Measures monthly price risk by calculating the standard deviation of closing prices grouped by ticker and calendar month. Higher values indicate greater price swings and therefore higher risk. Useful for portfolio risk assessment and month-over-month volatility comparison across tickers.

**_Why Materialized Views:_**

Unlike regular views that recompute on every query, materialized views store the result physically in Delta format and refresh on demand via CREATE OR REFRESH — making repeated dashboard queries fast and cheap.

In [0]:
%sql

--This Databricks SQL statement creates or refreshes a materialized view named 'workspace.gold.stock_daily_returns'.
--1-The daily percentage returns for each stock ticker.
--For each row, it computes the percent change in 'close' price compared to the previous day's 'close' for the same ticker.
--The calculation uses the LAG window function to access the prior day's close price, partitions by ticker, and orders by date.
--The result is rounded to 4 decimal places and stored as 'daily_return_pct'.
--The source data is from the 'stocks_bronze' table.

CREATE OR REFRESH MATERIALIZED VIEW workspace.gold.stock_daily_returns AS
SELECT
    ticker,
    date,
    close,
    ROUND(
        (close - LAG(close) OVER (PARTITION BY ticker ORDER BY date)) /
        LAG(close) OVER (PARTITION BY ticker ORDER BY date) * 100,
        4
    ) AS daily_return_pct
FROM stocks_silver;



--2. Moving Averages (trend analysis)
CREATE OR REFRESH MATERIALIZED VIEW workspace.gold.stocks_moving_averages AS
SELECT
    ticker,
    date,
    close,
    AVG(close) OVER (PARTITION BY ticker ORDER BY date ROWS BETWEEN 6 PRECEDING AND CURRENT ROW)  AS ma_7,
    AVG(close) OVER (PARTITION BY ticker ORDER BY date ROWS BETWEEN 29 PRECEDING AND CURRENT ROW) AS ma_30
FROM stocks_silver;



-- 3- 3. Volatility (risk measure)
CREATE OR REFRESH MATERIALIZED VIEW workspace.gold.stocks_volatility AS
SELECT
    ticker,
    DATE_TRUNC('month', date) AS month,
    ROUND(STDDEV(close), 2)   AS price_volatility
FROM stocks_silver
GROUP BY ticker, DATE_TRUNC('month', date);